# Replicating Rust (1987) with the NFXP estimator

Harold Zurcher's bus engine replacement data, the model of Class 9 and the nested fixed
point estimator of Class 10, put together to reproduce Table X of Rust (1987), check
that the estimates are what they should be, and compute the counterfactual the paper is
after: the demand for engine replacement as a function of its price.

Everything here runs on three files of this folder: `zurcher.py` (the model),
`nfxp.py` (the data reader and the estimator) and `busdata1234.csv` (the data).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from zurcher import zurcher
from nfxp import read_busdata, estim_zurcher

## Data

One row per bus-month: the mileage bin the bus is in, the decision taken there (1 =
replace) and the transition that led into the bin. The mileage is discretized the way
Rust's original code did it, with the cell index $k = \lceil m \cdot 175 / 450000 \rceil$
used as a one-based index into the grid, so the decision is evaluated at grid point
$k-1$, and the transition after a replacement is recorded from cell 0 to cell $k$, one
cell more than the model's own reset. This is the known quirk of Rust's data; the
paper's numbers are only reproduced with it. See the docstring of `read_busdata()`.

In [ ]:
data = read_busdata()                       # bus groups 1-4, 175 cells, 450 000 miles
print(data.head())
print(f'\n{len(data)} bus-months, {data.bus.nunique()} buses, {int(data.d.sum())} replacements')
print('mileage increments (in bins) and their counts:', np.bincount(data.dx).tolist())

## Estimation

`estim_zurcher` is the model with the data attached. `estimate()` runs Rust's three
stages: the transition probabilities from the increment frequencies (closed form), then
$(RC, c)$ by the partial likelihood with the transition probabilities fixed, then all
parameters jointly. The outer loop is BHHH with the analytical score, the inner loop is
the poly-algorithm and Newton–Kantorovich, warm started from the previous solution.

In [ ]:
est = estim_zurcher(data)                   # beta = 0.9999 by default
result = est.estimate()
print(result)

## Table X of the paper

Rust (1987), Table X: the linear cost function $c(x) = 0.001\,\theta_{11}\,x$ on the
175-cell grid, three samples, dynamic ($\beta = 0.9999$) and myopic ($\beta = 0$)
model. Standard errors in parentheses.

| $\beta$ | Groups | $N$ | $RC$ | $\theta_{11}$ | $\theta_{30}$ | $\theta_{31}$ | $\theta_{32}$ | $\theta_{33}$ | $\ell$ |
| :-- | :-- | --: | --: | --: | --: | --: | --: | --: | --: |
| .9999 | 1, 2, 3 | 3864 | 11.7257 (2.597) | 2.4569 (.9122) | .0937 | .4475 | .4459 | .0127 | −3993.991 |
| .9999 | 4 | 4292 | 10.896 (1.581) | 1.1732 (.327) | .1191 | .5762 | .2868 | .0158 | −4495.135 |
| .9999 | 1, 2, 3, 4 | 8156 | 9.7687 (1.226) | 1.3428 (.315) | .1071 | .5152 | .3621 | .0143 | −8607.889 |
| 0 | 1, 2, 3 | 3864 | 8.2969 (1.0477) | 56.1656 (13.4205) | .0937 | .4475 | .4459 | .0127 | −3996.353 |
| 0 | 4 | 4292 | 7.6423 (.7204) | 36.6692 (7.0675) | .1191 | .5762 | .2868 | .0158 | −4496.997 |
| 0 | 1, 2, 3, 4 | 8156 | 7.3113 (.5073) | 36.0175 (5.5145) | .1070 | .5152 | .3622 | .0143 | −8614.238 |

The same six estimations with our code, printed beside the paper. The one entry that
does not agree is $RC$ for group 4 at $\beta = .9999$, which the paper prints as 10.896:
every other number in that row matches 10.0896, so it is a misprint.

In [ ]:
PAPER = {   # (beta, groups): RC, se, c, se, log-likelihood
    (0.9999, (1, 2, 3)):    (11.7257, 2.597, 2.4569, 0.9122, -3993.991),
    (0.9999, (4,)):         (10.896, 1.581, 1.1732, 0.327, -4495.135),
    (0.9999, (1, 2, 3, 4)): (9.7687, 1.226, 1.3428, 0.315, -8607.889),
    (0.0, (1, 2, 3)):       (8.2969, 1.0477, 56.1656, 13.4205, -3996.353),
    (0.0, (4,)):            (7.6423, 0.7204, 36.6692, 7.0675, -4496.997),
    (0.0, (1, 2, 3, 4)):    (7.3113, 0.5073, 36.0175, 5.5145, -8614.238),
}
print(f'{"beta":>6} {"groups":<8} {"":<5} {"RC":>8} {"(se)":>7} {"c":>8} {"(se)":>8} {"loglik":>10}')
for (beta, groups), paper in PAPER.items():
    r = estim_zurcher(read_busdata(groups=groups), beta=beta).estimate()
    g = ','.join(map(str, groups))
    print(f'{beta:>6} {g:<8} {"ours":<5} {r.theta[0]:>8.4f} {r.se[0]:>7.3f} {r.theta[1]:>8.4f} {r.se[1]:>8.4f} {r.loglik:>10.3f}')
    print(f'{"":>6} {"":<8} {"paper":<5} {paper[0]:>8.4f} {paper[1]:>7.3f} {paper[2]:>8.4f} {paper[3]:>8.4f} {paper[4]:>10.3f}')
    assert abs(r.loglik - paper[4]) < 1e-3, 'log-likelihood differs from the paper'

## Basic diagnostics

Three checks that the estimator does what it claims, back at the full-sample dynamic
estimates.

**The analytical score.** The gradient of the log-likelihood comes from the implicit
function theorem applied to the fixed point. It must agree with finite differences of
the log-likelihood itself.

In [ ]:
theta = result.theta
analytical = est.score(theta).sum(axis=0)          # sum over bus-months
numerical = np.empty(2)
for k in range(2):                                 # RC and c; the p directions need a tiny step
    e = np.zeros(theta.size); e[k] = 1e-5
    numerical[k] = (est.loglik(theta + e) - est.loglik(theta - e)) / 2e-5
for name, a, n in zip(est.names, analytical, numerical):
    print(f'{name:<4} analytical {a:>12.6f}   finite difference {n:>12.6f}')

**Standard errors.** BHHH uses the outer product of the scores; the alternative is the
inverse of the negative Hessian, and the sandwich combines the two. The information
equality says they should agree if the model is right. They agree for the transition
probabilities and not for $RC$ and $c$: a sign of misspecification worth a discussion.

In [ ]:
V = est.standard_errors(theta)
print(f'{"":<4} {"BHHH":>10} {"Hessian":>10} {"sandwich":>10}')
for k, name in enumerate(est.names):
    print(f'{name:<4} ' + ' '.join(f'{np.sqrt(V[key][k, k]):>10.4f}' for key in ('bhhh', 'hessian', 'sandwich')))

**Starting values.** The likelihood is flat in $RC$, so an optimizer that stops early
can land anywhere within a hundredth of the estimate. From several starting points the
BHHH iterations should end at the same place.

In [ ]:
for start in ((0, 0), (20, 0.1), (1, 10), (30, 30)):
    r = estim_zurcher(data).estimate(theta0=start)
    print(f'start {str(start):<10} RC {r.theta[0]:.4f}  c {r.theta[1]:.4f}  loglik {r.loglik:.3f}  converged {r.converged}')
est.theta = theta                                  # back to the estimates

## Counterfactual: the demand for engine replacement

The model, solved at the estimates, is a controlled Markov chain on mileage. Its
stationary distribution $q(x)$ solves $q = q P$, where $P$ mixes the keep transition
with probability $P(\text{keep}|x)$ and the replace transition (a fresh engine, the
first row of the transition matrix) otherwise. The expected number of replacements per
bus per year is twelve times the stationary probability of a replacement.

In [ ]:
def ergodic_distribution(model, pk):
    '''Stationary distribution of mileage under the policy P(keep|x) = pk:
    returns q(x), q(x, keep), q(x, replace)'''
    P = pk[:, None] * model.trpr + (1 - pk)[:, None] * model.trpr[0][None, :]
    A = P.T - np.eye(model.n)
    A[-1, :] = 1.0                                 # one redundant equation replaced by sum(q) = 1
    b = np.zeros(model.n); b[-1] = 1.0
    q = np.maximum(np.linalg.solve(A, b), 0)
    q /= q.sum()
    return q, q * pk, q * (1 - pk)

_, pk, _ = est.solve()                             # the policy at the estimates
q, q_keep, q_replace = ergodic_distribution(est, pk)
miles = est.grid * est.max_miles / est.n / 1000    # grid points in thousands of miles
width = miles[1] - miles[0]
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(data.x * est.max_miles / est.n / 1000, bins=35, density=True, alpha=0.3, label='mileage in the data')
ax.plot(miles, q / width, lw=2, label='stationary distribution of mileage')
ax.plot(miles, q_replace / width, lw=2, label='… and of replacements')
ax.set_xlabel('mileage since last replacement, thousands of miles'); ax.set_ylabel('density')
ax.set_title(f'{12 * q_replace.sum():.3f} expected replacements per bus per year')
ax.legend(); plt.show()

The data sit to the left of the stationary distribution: most buses enter the sample
with a new engine, and the panel is too short to reach the steady state.

The demand curve re-solves the model on a grid of replacement costs and reads the
replacement rate off the stationary distribution each time. The cost is converted into
dollars with the average replacement cost of \$8062 for these bus groups (Table III of
the paper). Figure 7 of the paper draws it for the dynamic and for the myopic model; the
curves below do not reproduce that figure exactly, although the estimates they are
computed from reproduce the tables.

In [ ]:
def demand_curve(model, RC_grid):
    '''Expected annual replacements per bus as a function of RC; other parameters fixed'''
    saved, demand, ev = model.RC, np.empty(RC_grid.size), None
    for i, rc in enumerate(RC_grid):
        model.RC = rc
        ev, pk = model.solve_poly(ev0=ev, tol=1e-8)    # warm start along the grid
        demand[i] = 12 * ergodic_distribution(model, pk)[2].sum()
    model.RC = saved
    return demand

RC_grid = np.linspace(0.5, 30, 60)
myopic = estim_zurcher(data, beta=0.0)
myopic.estimate()
fig, ax = plt.subplots(figsize=(8, 5))
for m in (est, myopic):
    scale = 8062 / m.RC                            # dollars per unit of RC at the estimate
    ax.plot(RC_grid * scale, demand_curve(m, RC_grid), lw=2, label=f'beta = {m.beta}')
ax.axvline(8062, ls='--', color='gray', label='average replacement cost')
ax.set_xlim(0, 12000); ax.set_ylim(0, 1)
ax.set_xlabel('replacement cost, dollars'); ax.set_ylabel('expected engine replacements per bus per year')
ax.set_title('Expected demand for engine replacement, one bus'); ax.grid(alpha=0.3); ax.legend(); plt.show()

h = 0.01
d = demand_curve(est, est.RC * np.array([1 - h, 1 + h]))
print(f'Demand elasticity with respect to RC at the estimate: {(d[1] - d[0]) / (2 * h) / d.mean():.3f}')

The two models fit the data equally well at the observed price and disagree everywhere
else: the myopic bus manager reacts to a price change far more, because in his model
the only reason to replace is today's cost. The discount factor matters for the
counterfactual even where the likelihood can barely tell $\beta = 0.9999$ from
$\beta = 0$.